In [2]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [5]:
# 1. load encoder (already loaded from pretrained BERT)
# 2. load decoder from stage 1
checkpoint = torch.load("stage1_best.pt", map_location=device)
decoder.load_state_dict(checkpoint["decoder"])
for param in decoder.parameters():
    param.requires_grad = False
print("stage1 decoder loaded and frozen")

# 3. init new stage 2 components fresh
pooler     = SequencePooler(hidden_dim=768, latent_dim=256).to(device)
metric_net = MetricNet(latent_dim=256).to(device)
flow_net   = FlowNet(latent_dim=256).to(device)

# 4. optimizer only touches stage 2 components
optimizer = AdamW(
    list(pooler.parameters()) +
    list(metric_net.parameters()) +
    list(flow_net.parameters()),
    lr=1e-4
)

FileNotFoundError: [Errno 2] No such file or directory: 'stage1_best.pt'

In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class MetricNet(nn.Module):
    def __init__(self, latent_dim=256):
        super().__init__()
        self.latent_dim = latent_dim
        self.net = nn.Sequential(
            nn.Linear(latent_dim, 512),
            nn.SiLU(),
            nn.Linear(512, 512),
            nn.SiLU(),
            nn.Linear(512, latent_dim * latent_dim)
        )
    
    def forward(self, z):
        B = z.shape[0]
        L_flat = self.net(z)                                    # [B, d*d]
        L = L_flat.view(B, self.latent_dim, self.latent_dim)   # [B, d, d]
        
        # enforce lower triangular
        L = torch.tril(L)
        
        # enforce positive diagonal via softplus
        diag_idx = torch.arange(self.latent_dim)
        L = L.clone()
        L[:, diag_idx, diag_idx] = F.softplus(L[:, diag_idx, diag_idx]) + 1e-4
        
        # SPD matrix
        G = L @ L.transpose(-2, -1)                            # [B, d, d]
        return G, L


# ── test ──────────────────────────────────────────────────────────────
metric_net = MetricNet(latent_dim=256).to(device)

z_test = torch.randn(2, 256).to(device)
G, L = metric_net(z_test)

eigenvalues = torch.linalg.eigvalsh(G)

print(f"G shape:        {G.shape}")           # [2, 256, 256]
print(f"L shape:        {L.shape}")           # [2, 256, 256]
print(f"min eigenvalue: {eigenvalues.min().item():.6f}")  # must be > 0
print(f"max eigenvalue: {eigenvalues.max().item():.6f}")
print(f"SPD check:      {'PASSED' if eigenvalues.min() > 0 else 'FAILED'}")

G shape:        torch.Size([2, 256, 256])
L shape:        torch.Size([2, 256, 256])
min eigenvalue: 0.009496
max eigenvalue: 3.867041
SPD check:      PASSED


In [4]:
class FlowNet(nn.Module):
    def __init__(self, latent_dim=256):
        super().__init__()
        self.latent_dim = latent_dim
        self.net = nn.Sequential(
            nn.Linear(latent_dim + 1, 512),  # +1 for time t
            nn.SiLU(),
            nn.Linear(512, 512),
            nn.SiLU(),
            nn.Linear(512, 512),
            nn.SiLU(),
            nn.Linear(512, latent_dim)
        )
    
    def forward(self, z_t, t):
        # z_t: [B, latent_dim]
        # t:   [B]
        t_expand = t.unsqueeze(-1)                  # [B, 1]
        inp      = torch.cat([z_t, t_expand], dim=-1)  # [B, latent_dim+1]
        return self.net(inp)                        # [B, latent_dim]


# ── test ──────────────────────────────────────────────────────────────
flow_net = FlowNet(latent_dim=256).to(device)

z_t_test = torch.randn(2, 256).to(device)
t_test   = torch.rand(2).to(device)

v_pred = flow_net(z_t_test, t_test)

print(f"z_t shape: {z_t_test.shape}")  # [2, 256]
print(f"t shape:   {t_test.shape}")    # [2]
print(f"v_pred shape: {v_pred.shape}") # [2, 256]
print("FlowNet check: PASSED" if v_pred.shape == z_t_test.shape else "FAILED")

z_t shape: torch.Size([2, 256])
t shape:   torch.Size([2])
v_pred shape: torch.Size([2, 256])
FlowNet check: PASSED


In [ ]:
import ot
from torch.optim import AdamW

# ── pooling layer to go from sequence to single vector ─────────────
class SequencePooler(nn.Module):
    def __init__(self, hidden_dim=768, latent_dim=256):
        super().__init__()
        self.project_down = nn.Linear(hidden_dim, latent_dim)
    
    def forward(self, hidden_states):
        # hidden_states: [B, seq_len, 768]
        pooled = hidden_states.mean(dim=1)  # [B, 768]
        z = self.project_down(pooled)        # [B, 256]
        return z

# ── OT pairing ─────────────────────────────────────────────────────
def get_ot_pairs(z_noise, z_data):
    cost = torch.cdist(z_noise, z_data, p=2).pow(2)
    a = torch.ones(z_noise.shape[0]) / z_noise.shape[0]
    b = torch.ones(z_data.shape[0])  / z_data.shape[0]
    T = ot.emd(
        a.cpu().numpy(),
        b.cpu().numpy(),
        cost.detach().cpu().numpy()
    )
    T_tensor = torch.from_numpy(T).to(z_noise.device)
    T_flat   = T_tensor.flatten()
    indices  = torch.multinomial(T_flat, z_noise.shape[0], replacement=True)
    i = indices // z_noise.shape[0]
    j = indices  % z_noise.shape[0]
    return z_noise[i], z_data[j]

# ── init all components ─────────────────────────────────────────────
pooler     = SequencePooler(hidden_dim=768, latent_dim=256).to(device)
metric_net = MetricNet(latent_dim=256).to(device)
flow_net   = FlowNet(latent_dim=256).to(device)

optimizer = AdamW(
    list(pooler.parameters()) +
    list(metric_net.parameters()) +
    list(flow_net.parameters()),
    lr=1e-4
)

# ── training loop ───────────────────────────────────────────────────
EPOCHS     = 5
best_loss  = float("inf")

for epoch in range(EPOCHS):
    metric_net.train()
    flow_net.train()
    pooler.train()
    encoder.eval()

    train_loss = 0
    fm_loss_total     = 0
    metric_loss_total = 0

    for step, batch in enumerate(train_loader):
        input_ids      = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        # encode → pool → single latent vector
        with torch.no_grad():
            hidden = encoder(input_ids, attention_mask)  # [B, 128, 768]
        z_data = pooler(hidden)                          # [B, 256]

        # sample noise + OT pairing
        z_noise_raw      = torch.randn_like(z_data)
        z_noise, z_data  = get_ot_pairs(z_noise_raw, z_data)

        # sample time
        t = torch.rand(z_data.shape[0]).to(device)

        # interpolate along path
        z_t   = (1 - t.unsqueeze(-1)) * z_noise + t.unsqueeze(-1) * z_data

        # true vector field
        v_true = z_data - z_noise                        # [B, 256]

        # metric at z_t
        G, L = metric_net(z_t)                           # [B, 256, 256]

        # predicted vector field
        v_pred = flow_net(z_t, t)                        # [B, 256]

        # fm loss — weighted by metric
        diff        = (v_pred - v_true).unsqueeze(-1)    # [B, 256, 1]
        fm_loss     = (diff.transpose(-2,-1) @ G @ diff).mean()

        # metric regularization
        eigenvalues = torch.linalg.eigvalsh(G)
        metric_loss = (
            torch.relu(1e-4 - eigenvalues).mean() +
            torch.relu(eigenvalues - 100.0).mean()
        )

        total_loss = fm_loss + 0.01 * metric_loss

        optimizer.zero_grad()
        total_loss.backward()
        torch.nn.utils.clip_grad_norm_(
            list(metric_net.parameters()) +
            list(flow_net.parameters()) +
            list(pooler.parameters()),
            max_norm=1.0
        )
        optimizer.step()

        train_loss        += total_loss.item()
        fm_loss_total     += fm_loss.item()
        metric_loss_total += metric_loss.item()

        if step % 50 == 0:
            print(f"epoch {epoch+1} step {step}/{len(train_loader)} | "
                  f"total {total_loss.item():.4f} | "
                  f"fm {fm_loss.item():.4f} | "
                  f"metric {metric_loss.item():.6f}")

    avg_total  = train_loss        / len(train_loader)
    avg_fm     = fm_loss_total     / len(train_loader)
    avg_metric = metric_loss_total / len(train_loader)

    print(f"\nepoch {epoch+1} done | "
          f"total {avg_total:.4f} | "
          f"fm {avg_fm:.4f} | "
          f"metric {avg_metric:.6f}\n")

    if avg_total < best_loss:
        best_loss = avg_total
        torch.save({
            "pooler":      pooler.state_dict(),
            "metric_net":  metric_net.state_dict(),
            "flow_net":    flow_net.state_dict(),
        }, "stage2_best.pt")
        print(f"saved best model at loss {best_loss:.4f}")